# 016 Synthetic Run Assembly

Assembles a synthetic combined run from two sources:
- **temp/**: new R=30 runs (priority) + some R=50 runs, stored as ZIPs per config
- **databricks_full_run1/**: older R=50 runs

Goal: 30 unique task IDs per dataset per config (W1_fc, W2_fc, W5_fc), with the **same task IDs across configs**.

Priority:
1. R=30 runs first (clean)
2. Fill gaps by random-sampling 30 reps from R=50 runs (seed=42)
3. Task IDs identical across W1/W2/W5

In [ ]:
import json
import zipfile
import re
import random
import shutil
from pathlib import Path
from collections import defaultdict

SEED = 42
TARGET = 30
CONFIGS = ['W1_fc', 'W2_fc', 'W5_fc']
DATASETS = ['hiddenbench', 'gpqa']

ROOT = Path('/Users/I550854/Documents/Master Thesis/self-organization-mas')
TEMP_DIR = ROOT / 'results' / 'temp'
OLD_DIR = ROOT / 'results' / 'mas' / 'databricks_full_run1'

def parse_fname(fname):
    m = re.search(r'(hiddenbench|gpqa)_.*?_(q\d+)_R(\d+)', fname)
    if m:
        return m.group(1), m.group(2), int(m.group(3))
    return None, None, None

## Cell 1: Count available tasks per source

In [ ]:
def collect_entries(config):
    entries = []
    seen = set()

    zip_path = TEMP_DIR / f'{config} (1).zip'
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if not name.endswith('.json'):
                continue
            fname = Path(name).name
            dataset, task_id, R = parse_fname(fname)
            if dataset is None:
                continue
            key = (dataset, task_id, R)
            if key in seen:
                continue
            seen.add(key)
            entries.append({
                'dataset': dataset, 'task_id': task_id, 'R': R,
                'source': 'temp_zip', 'zip_path': zip_path, 'zip_name': name, 'fname': fname,
            })

    old_config_dir = OLD_DIR / config
    for fpath in sorted(old_config_dir.glob('*.json')):
        dataset, task_id, R = parse_fname(fpath.name)
        if dataset is None:
            continue
        key = (dataset, task_id, R)
        if key in seen:
            continue
        seen.add(key)
        entries.append({
            'dataset': dataset, 'task_id': task_id, 'R': R,
            'source': 'old_dir', 'file_path': fpath, 'fname': fpath.name,
        })

    return entries

print(f'{"Config":<10} {"Dataset":<15} {"R=30 tasks":<15} {"R=50 tasks":<15} {"Total unique"}')
print('-' * 65)

all_entries = {}
for config in CONFIGS:
    entries = collect_entries(config)
    all_entries[config] = entries
    for dataset in DATASETS:
        sub = [e for e in entries if e['dataset'] == dataset]
        r30 = {e['task_id'] for e in sub if e['R'] == 30}
        r50 = {e['task_id'] for e in sub if e['R'] == 50}
        unique = {e['task_id'] for e in sub}
        print(f'{config:<10} {dataset:<15} {len(r30):<15} {len(r50):<15} {len(unique)}')

## Cell 2: Determine shared task IDs and sample to reach TARGET=35

In [ ]:
rng = random.Random(SEED)

def pick_best_entry(entries_for_task):
    r30 = [e for e in entries_for_task if e['R'] == 30]
    if r30:
        return r30[0]
    return entries_for_task[0]

def select_tasks_for_dataset(config_entries_map, dataset, target=TARGET):
    per_config_tasks = {}
    for config, entries in config_entries_map.items():
        sub = [e for e in entries if e['dataset'] == dataset]
        by_task = defaultdict(list)
        for e in sub:
            by_task[e['task_id']].append(e)
        per_config_tasks[config] = by_task

    shared_all = set.intersection(*[set(t.keys()) for t in per_config_tasks.values()])
    r30_in_all = {tid for tid in shared_all if all(any(e['R'] == 30 for e in per_config_tasks[c][tid]) for c in CONFIGS)}
    r50_only_shared = shared_all - r30_in_all

    selected = sorted(r30_in_all)
    if len(selected) < target:
        remaining = sorted(r50_only_shared)
        rng.shuffle(remaining)
        needed = target - len(selected)
        selected += remaining[:needed]
    selected = selected[:target]

    result = {}
    for tid in selected:
        result[tid] = {}
        for config in CONFIGS:
            result[tid][config] = pick_best_entry(per_config_tasks[config][tid])
    return result

selected_by_dataset = {}
for dataset in DATASETS:
    selected = select_tasks_for_dataset(all_entries, dataset)
    selected_by_dataset[dataset] = selected

    r30_count = sum(1 for tid, cm in selected.items() for c, e in cm.items() if e['R'] == 30)
    total_entries = len(selected) * len(CONFIGS)
    print(f'\n=== {dataset} ===')
    print(f'Selected {len(selected)}/{TARGET} task IDs')
    print(f'R=30 entries: {r30_count}/{total_entries},  R=50 entries: {total_entries - r30_count}/{total_entries}')

    for tid in sorted(selected.keys(), key=lambda x: int(x[1:])):
        row = [f'{c}:R{selected[tid][c]["R"]}({selected[tid][c]["source"][0]})' for c in CONFIGS]
        print(f'  {tid:<8} {"  ".join(row)}')

## Cell 3: Save synthetic run

In [ ]:
OUT_DIR = ROOT / 'results' / 'mas' / 'synthetic_run_fc_30tasks'

def read_entry(entry):
    if entry['source'] == 'temp_zip':
        with zipfile.ZipFile(entry['zip_path']) as zf:
            with zf.open(entry['zip_name']) as f:
                return json.load(f)
    else:
        with open(entry['file_path']) as f:
            return json.load(f)

def make_output_fname(data, config):
    w = config[1]
    dataset = data['dataset']
    qid = data['question_id']
    R = len(data['repetitions'])
    return f"{dataset}_W{w}_topofc_q{qid}_R{R}_synthetic.json"

saved = defaultdict(int)
skipped = []

for config in CONFIGS:
    config_out = OUT_DIR / config
    config_out.mkdir(parents=True, exist_ok=True)

for dataset, task_map in selected_by_dataset.items():
    for tid, cfg_map in task_map.items():
        for config, entry in cfg_map.items():
            try:
                data = read_entry(entry)

                if entry['R'] == 50:
                    reps = data['repetitions']
                    rng_rep = random.Random(SEED + int(tid[1:]))
                    sampled_reps = rng_rep.sample(reps, 30)
                    data['repetitions'] = sampled_reps
                    data['R'] = 30
                    data['_synthetic_note'] = f'sampled 30 from R=50, seed={SEED + int(tid[1:])}'
                else:
                    data['_synthetic_note'] = 'original R=30 run'

                out_fname = make_output_fname(data, config)
                out_path = OUT_DIR / config / out_fname
                with open(out_path, 'w') as f:
                    json.dump(data, f)
                saved[config] += 1
            except Exception as exc:
                skipped.append((config, tid, dataset, str(exc)))

print(f'Output: {OUT_DIR}')
print()
for config in CONFIGS:
    files = list((OUT_DIR / config).glob('*.json'))
    by_ds = defaultdict(list)
    for f in files:
        ds = 'gpqa' if 'gpqa' in f.name else 'hiddenbench'
        by_ds[ds].append(f.name)
    print(f'{config}: {len(files)} files total')
    for ds, fnames in by_ds.items():
        print(f'  {ds}: {len(fnames)}')

if skipped:
    print(f'\nSKIPPED ({len(skipped)}):')
    for s in skipped:
        print(f'  {s}')
else:
    print('\nNo errors.')